# Structure, Diffraction, And Visualization Pipeline

This notebook shows the architectural point of the new features: one `Phase` can
feed structure visualization, powder XRD, and SAED workflows without redefining the
crystallographic semantics at each step. The immediate roadmap path now uses the
pinned `ni_fcc` fixture and reads the manifest-backed reproducibility trail that
supports the same example in tests and docs.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

scene = build_crystal_scene(
    phase,
    repeats=(2, 2, 2),
    show_unit_cells=True,
    plane_overlays=(
        CrystalPlaneOverlay(
            plane=CrystalPlane(MillerIndex([1, 1, 1], phase=phase), phase=phase),
            label_indices=(1, 1, 1),
            color="#f97316",
            alpha=0.20,
        ),
    ),
    direction_overlays=(
        CrystalDirectionOverlay(
            direction=CrystalDirection(np.array([1.0, 1.0, 0.0]), phase=phase),
            anchor_fractional=np.array([0.0, 0.0, 0.0]),
            label_indices=(1, 1, 0),
            color="#2563eb",
        ),
    ),
    style_overrides=publication_crystal_style(),
)
xrd = generate_xrd_pattern(phase, broadening_fwhm_deg=0.16, max_index=6)
saed = generate_saed_pattern(
    phase,
    ZoneAxis(indices=np.array([1, 1, 0]), phase=phase),
    camera_constant_mm_angstrom=180.0,
    max_index=5,
    max_g_inv_angstrom=3.5,
)

print("scene atoms:", len(scene.atoms))
print("xrd reflections:", len(xrd.reflections))
print("saed spots:", len(saed.spots))


In [ ]:
structure_audit = read_workflow_result_manifest(
    "benchmarks/structure_import/foundation_workflow_result_manifest.json"
).to_dict()
diffraction_baselines = read_workflow_result_manifest(
    "benchmarks/diffraction/external_baseline_workflow_result_manifest.json"
).to_dict()
diffraction_validation = read_validation_manifest(
    "benchmarks/validation/diffraction_validation_manifest.json"
).to_dict()

print("Structure audit result:", structure_audit["result_id"])
print("Audit summary:", structure_audit["metadata"]["audit_summary_artifact"])
print(
    "External baseline packages:",
    diffraction_baselines["metadata"]["xrd_reference_package"],
    diffraction_baselines["metadata"]["saed_reference_package"],
)
print(
    "Validation campaign:",
    diffraction_validation["campaign_name"],
    diffraction_validation["status"],
)


In [ ]:
xrd_figure = plot_xrd_pattern(xrd, theme="journal")
saed_figure = plot_saed_pattern(saed, theme="dark")
crystal_figure = plot_crystal_structure_3d(
    scene,
    projection="persp",
    style_overrides=publication_crystal_style(),
)
xrd_figure


In [ ]:
saed_figure


In [ ]:
crystal_figure


This is the main fixture-backed teaching path for the current immediate roadmap:
pinned phase fixture -> canonical structure model -> runtime visualization and
diffraction -> manifest-backed validation artifacts.

The figures are stable teaching and publication surfaces for the current
implementation, but the XRD and SAED intensity models remain intentionally modest
compared with full physical scattering engines.
